<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/20_GES_Aware_Genomic_RAG_Cell_7C13_Unblinding_and_Routing_Authorization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm Google Drive is mounted and the project directory is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, exact Cell 7C12 lineage, and fail-closed output paths

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import hashlib
import json
import re
import tempfile

import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = (
    '20_GES_Aware_Genomic_RAG_Cell_7C13_'
    'Unblinding_and_Routing_Authorization.ipynb'
)
CELL_ID = '7C13'
STAGE = '7C'
AMENDMENT_ID = 'A004'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_RESPONSES = 1_440
EXPECTED_QUESTIONS = 80

EXPECTED_CELL_7C12_TERMINAL_DECISION = (
    'PASS_STAGE7C12_A004_BLINDED_AUTOMATED_RESPONSE_LEVEL_SCORING_COMPLETE_'
    '1440_OF_1440_FOUR_PRIMARY_COMPONENTS_AND_AUTOMATED_EVIDENCE_FIDELITY_'
    'COMPOSITE_FROZEN_CHECKSUM_PROTECTED_SCORE_BLIND_NO_HUMAN_REVIEW_FREE_TEXT_'
    'FACTUAL_CORRECTNESS_OR_SEMANTIC_CITATION_ENTAILMENT_CLAIM_NO_CONDITION_'
    'UNBLINDING_ROUTING_ACCESS_RUN_AGGREGATION_BOOTSTRAP_OR_ARM_COMPARISON_'
    'NEXT_AUTOMATED_EXECUTION_NOT_AUTHORIZED'
)

# --------------------------------------------------------------------------------------
# Exact successful Cell 7C12 package.
# --------------------------------------------------------------------------------------
CELL_7C12_DATA_DIR = (
    ROOT / 'data_processed' / 'stage7_rag'
    / 'cell_7c12_a004_blinded_automated_response_level_outcomes_v1'
)
CELL_7C12_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c12_a004_blinded_automated_response_level_outcomes_v1'
)
CELL_7C12_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c12_a004_blinded_automated_response_level_outcomes_v1'
)

CELL_7C12 = OrderedDict([
    ('response_level_outcomes', {
        'path': CELL_7C12_DATA_DIR / 'cell_7c12_a004_blinded_automated_response_level_outcomes_v1.parquet',
        'sha256': '53b48a7cb59d3af03fa48444b681e8b324cb5def5ce94f193f42d2ee8725a147',
    }),
    ('endpoint_derivation_schema', {
        'path': CELL_7C12_CONFIG_DIR / 'cell_7c12_a004_endpoint_derivation_schema_v1.json',
        'sha256': 'b578779e9b3c15112cee5e4149f6efc3d06ebc29063b2c1805e5d40d2e3bcabd',
    }),
    ('input_inventory', {
        'path': CELL_7C12_CONFIG_DIR / 'cell_7c12_verified_input_inventory_v1.csv',
        'sha256': 'fe6ce2fd839fa89dcff064c9493b62909dbe8223cf3b9aba022b87841339c9b7',
    }),
    ('execution_report', {
        'path': CELL_7C12_QC_DIR / 'cell_7c12_a004_blinded_scoring_execution_report_v1.json',
        'sha256': '6a73b53551be09789b6f9a677dcfd533d2a93d7fa4a4ecd10de5b6e508f18c42',
    }),
    ('qc', {
        'path': CELL_7C12_QC_DIR / 'cell_7c12_a004_blinded_scoring_qc_v1.json',
        'sha256': '06ca1f1bac0d57ca4e149d89eaf41466a51aca10c1fa176c4f65ed6ae9aa5937',
    }),
    ('manifest', {
        'path': CELL_7C12_CONFIG_DIR / 'cell_7c12_a004_blinded_scoring_manifest_v1.json',
        'sha256': 'dab5898271ecf7d8304c210afcdf5668e2ada4c6facc86a3d1bb28b23e9de70a',
    }),
])

# --------------------------------------------------------------------------------------
# Frozen A004 aggregation/inference specification.
# --------------------------------------------------------------------------------------
A004_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'protocol_amendment_A004_fully_automated_structured_evaluation_v1'
)
A004_AGGREGATION_SPEC = {
    'path': A004_DIR / 'protocol_amendment_A004_aggregation_and_inference_spec_v1.json',
    'sha256': 'b4ea6ae437edfbdc626d249855203c2b6f9b2b6131212d48424760cdeaf6d1f9',
}

# --------------------------------------------------------------------------------------
# Exact internal routing map from Cell 7C8. Cell 7C13 verifies but DOES NOT OPEN it.
# --------------------------------------------------------------------------------------
CELL_7C8_ROUTING_MAP = {
    'path': (
        ROOT / 'configs' / 'stage7_rag'
        / 'cell_7c8_blinded_reviewer_packet_v1'
        / 'cell_7c8_internal_blinded_review_routing_map_v1.parquet'
    ),
    'sha256': '8c65375e6a24761bd14e2d59d837507c67146ed88e8e64a533bca304598c696c',
}

# Frozen 7B4 alias mapping SHA-256. Path is located by hash only; contents remain unopened in Cell 7C13.
FROZEN_7B4_ALIAS_MAPPING_SHA256 = (
    '6eb45683b42a456d2b6788a5fcf6b9cd95fc606afe9627610ebbc11914312cb9'
)

CONFIG_SEARCH_ROOT = ROOT / 'configs' / 'stage7_rag'

OUT_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c13_unblinding_and_routing_authorization_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c13_unblinding_and_routing_authorization_v1'
)

OUTPUTS = OrderedDict([
    ('authorization',
     OUT_DIR / 'cell_7c13_unblinding_and_routing_authorization_v1.json'),
    ('verified_input_inventory',
     OUT_DIR / 'cell_7c13_verified_input_inventory_v1.csv'),
    ('qc',
     QC_DIR / 'cell_7c13_unblinding_authorization_qc_v1.json'),
    ('manifest',
     OUT_DIR / 'cell_7c13_unblinding_authorization_manifest_v1.json'),
])

for directory in (OUT_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing:
    raise FileExistsError(
        'Cell 7C13 fail-closed overwrite protection is active. Existing output(s):\\n- '
        + '\\n- '.join(existing)
    )

print(f'Authorization directory: {OUT_DIR}')
print(f'QC directory           : {QC_DIR}')

Authorization directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c13_unblinding_and_routing_authorization_v1
QC directory           : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c13_unblinding_and_routing_authorization_v1


## 2. SHA-256, sidecar, and stable-write helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid SHA-256 sidecar: {path}')
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    return (
        path.exists()
        and sidecar_path(path).exists()
        and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)
    )


def verify_exact_artifact(
    label: str,
    path: Path,
    expected_sha256: str,
    require_sidecar: bool = True,
) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'{label} SHA-256 mismatch.\\nExpected: {expected_sha256}\\nObserved: {observed}'
        )
    if require_sidecar and not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)) if sidecar_path(path).exists() else '',
        'sidecar_valid': sidecar_is_valid(path) if sidecar_path(path).exists() else False,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def stable_write_json(path: Path, payload: Any) -> str:
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        ) + chr(10),
        encoding='utf-8',
    )
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    sidecar_path(path).write_text(
        f'{sha256_file(path)}  {path.name}' + chr(10),
        encoding='utf-8',
    )


def find_unique_file_by_sha256(search_root: Path, target_sha256: str) -> Path:
    if not search_root.exists():
        raise FileNotFoundError(f'Search root not found: {search_root}')

    matches = []
    skipped_suffixes = {'.sha256', '.lock'}

    for path in sorted(search_root.rglob('*')):
        if not path.is_file():
            continue
        if path.suffix.lower() in skipped_suffixes:
            continue
        try:
            if sha256_file(path) == target_sha256:
                matches.append(path)
        except OSError:
            continue

    if len(matches) != 1:
        raise RuntimeError(
            f'Expected exactly one config artifact with SHA-256 {target_sha256}; '
            f'found {len(matches)}: {[str(x) for x in matches]}'
        )

    return matches[0]


with tempfile.TemporaryDirectory(prefix='cell_7c13_writer_test_') as tmp:
    p = Path(tmp) / 'x.json'
    stable_write_json(p, {'ok': True})
    write_sidecar(p)
    assert load_json(p) == {'ok': True}
    assert sidecar_is_valid(p)

print('Serialization / SHA-256 helper self-test: PASS')

Serialization / SHA-256 helper self-test: PASS


## 3. Reverify the complete frozen Cell 7C12 outcome package

In [4]:
verified_inputs = []

for artifact_id, spec in CELL_7C12.items():
    record = verify_exact_artifact(
        f'cell_7c12_{artifact_id}',
        spec['path'],
        spec['sha256'],
        require_sidecar=True,
    )
    record['source_cell'] = '7C12'
    verified_inputs.append(record)

manifest_7c12 = load_json(CELL_7C12['manifest']['path'])
qc_7c12 = load_json(CELL_7C12['qc']['path'])
report_7c12 = load_json(CELL_7C12['execution_report']['path'])

if manifest_7c12.get('terminal_decision') != EXPECTED_CELL_7C12_TERMINAL_DECISION:
    raise AssertionError('Cell 7C12 terminal PASS mismatch.')
if manifest_7c12.get('next_authorized_cell') is not None:
    raise AssertionError('Cell 7C12 unexpectedly authorized a downstream cell.')
if manifest_7c12.get('condition_identity_unblinding_authorized') is not False:
    raise AssertionError('Cell 7C12 unexpectedly authorized condition unblinding.')
if manifest_7c12.get('internal_routing_map_access_authorized') is not False:
    raise AssertionError('Cell 7C12 unexpectedly authorized routing-map access.')
if manifest_7c12.get('run_aggregation_authorized') is not False:
    raise AssertionError('Cell 7C12 unexpectedly authorized run aggregation.')
if manifest_7c12.get('bootstrap_inference_authorized') is not False:
    raise AssertionError('Cell 7C12 unexpectedly authorized bootstrap inference.')
if manifest_7c12.get('arm_comparison_authorized') is not False:
    raise AssertionError('Cell 7C12 unexpectedly authorized arm comparison.')
if int(qc_7c12.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C12 QC does not report zero failures.')

outcome_pf = pq.ParquetFile(CELL_7C12['response_level_outcomes']['path'])
if int(outcome_pf.metadata.num_rows) != EXPECTED_RESPONSES:
    raise AssertionError('Cell 7C12 frozen outcome table does not contain 1,440 rows.')

outcome_schema = set(outcome_pf.schema_arrow.names)
for prohibited in ['blinded_alias', 'run_id', 'condition_id', 'condition_name']:
    if prohibited in outcome_schema:
        raise AssertionError(
            f'Cell 7C12 outcome table is unexpectedly already unblinded: {prohibited}'
        )

print('Cell 7C12 package                     : 6/6 exact hashes + sidecars')
print('Cell 7C12 terminal PASS               : VERIFIED')
print('Frozen response-level outcomes        : 1,440')
print('Condition identity in outcome table   : ABSENT')
print('Run ID in outcome table               : ABSENT')

Cell 7C12 package                     : 6/6 exact hashes + sidecars
Cell 7C12 terminal PASS               : VERIFIED
Frozen response-level outcomes        : 1,440
Condition identity in outcome table   : ABSENT
Run ID in outcome table               : ABSENT


## 4. Verify the future-analysis specification and routing artifacts without opening routing contents

In [5]:
# Frozen aggregation/inference specification is safe to read: it contains the prespecified analysis design,
# not per-response condition restoration.
agg_record = verify_exact_artifact(
    'cell_7c11_A004_aggregation_inference_spec',
    A004_AGGREGATION_SPEC['path'],
    A004_AGGREGATION_SPEC['sha256'],
    require_sidecar=True,
)
agg_record['source_cell'] = '7C11'
verified_inputs.append(agg_record)

aggregation_spec = load_json(A004_AGGREGATION_SPEC['path'])

if aggregation_spec['primary_comparison']['experimental_condition'] != 'D Full-GES':
    raise AssertionError('Frozen primary experimental condition changed.')
if aggregation_spec['primary_comparison']['reference_condition'] != 'A semantic-only':
    raise AssertionError('Frozen primary reference condition changed.')
if aggregation_spec['bootstrap']['replicates'] != 2000:
    raise AssertionError('Frozen bootstrap replicate count changed.')
if len(aggregation_spec['mandatory_secondary_comparisons']) != 4:
    raise AssertionError('Frozen mandatory secondary comparison family changed.')

# Routing map: checksum only. Do not use pandas/pyarrow to open contents here.
routing_record = verify_exact_artifact(
    'cell_7c8_internal_blinded_review_routing_map',
    CELL_7C8_ROUTING_MAP['path'],
    CELL_7C8_ROUTING_MAP['sha256'],
    require_sidecar=True,
)
routing_record['source_cell'] = '7C8'
verified_inputs.append(routing_record)

# Find exact alias mapping by frozen SHA only. Contents remain unopened.
alias_mapping_path = find_unique_file_by_sha256(
    CONFIG_SEARCH_ROOT,
    FROZEN_7B4_ALIAS_MAPPING_SHA256,
)

# The historical config artifact may or may not have a sidecar depending on its original freeze package.
alias_record = verify_exact_artifact(
    'cell_7b4_frozen_blinded_alias_mapping',
    alias_mapping_path,
    FROZEN_7B4_ALIAS_MAPPING_SHA256,
    require_sidecar=False,
)
alias_record['source_cell'] = '7B4'
verified_inputs.append(alias_record)

print('A004 aggregation/inference spec        : VERIFIED')
print('Primary comparison frozen             : D vs A')
print('Mandatory secondary family            : D vs B/C/E/F')
print('Bootstrap replicates                  : 2,000')
print('Cell 7C8 routing map SHA              : VERIFIED — CONTENTS NOT OPENED')
print('Frozen 7B4 alias mapping SHA          : VERIFIED — CONTENTS NOT OPENED')
print(f'Frozen alias mapping artifact          : {alias_mapping_path}')

A004 aggregation/inference spec        : VERIFIED
Primary comparison frozen             : D vs A
Mandatory secondary family            : D vs B/C/E/F
Bootstrap replicates                  : 2,000
Cell 7C8 routing map SHA              : VERIFIED — CONTENTS NOT OPENED
Frozen 7B4 alias mapping SHA          : VERIFIED — CONTENTS NOT OPENED
Frozen alias mapping artifact          : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7b4_configuration_freeze_v1/cell_7b4_condition_alias_inventory_v1.csv


## 5. Freeze authorization for Cell 7C14 condition restoration only

In [6]:
authorization_decision = (
    'AUTHORIZE_STAGE7C_CELL7C14_CONDITION_RESTORATION_ONLY_BY_JOINING_1440_FROZEN_'
    'CELL7C12_RESPONSE_LEVEL_OUTCOMES_TO_EXACT_CELL7C8_INTERNAL_ROUTING_MAP_AND_'
    'EXACT_FROZEN_7B4_BLINDED_ALIAS_MAPPING_RESTORE_BLINDED_ALIAS_RUN_ID_AND_'
    'EXPERIMENTAL_CONDITION_IDENTITY_FREEZE_UNBLINDED_RESPONSE_LEVEL_TABLE_NO_'
    'RUN_AGGREGATION_CONDITION_LEVEL_PERFORMANCE_PRIMARY_OR_SECONDARY_ARM_'
    'COMPARISON_BOOTSTRAP_OR_INFERENCE'
)

authorization_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'version': '1.0.0',
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'authorization_decision': authorization_decision,
    'authorization_basis': {
        'cell_7c12_response_level_outcomes_sha256':
            CELL_7C12['response_level_outcomes']['sha256'],
        'cell_7c12_manifest_sha256':
            CELL_7C12['manifest']['sha256'],
        'cell_7c8_internal_routing_map_sha256':
            CELL_7C8_ROUTING_MAP['sha256'],
        'frozen_7b4_alias_mapping_sha256':
            FROZEN_7B4_ALIAS_MAPPING_SHA256,
        'frozen_7b4_alias_mapping_path':
            str(alias_mapping_path),
        'A004_aggregation_inference_spec_sha256':
            A004_AGGREGATION_SPEC['sha256'],
    },
    'cell_7c13_operations': {
        'routing_map_contents_opened': False,
        'alias_mapping_contents_opened': False,
        'condition_identity_restored': False,
        'run_aggregation_performed': False,
        'arm_performance_calculated': False,
        'bootstrap_performed': False,
    },
    'next_authorized_cell': '7C14',
    'cell_7c14_scope': {
        'routing_map_access_authorized': True,
        'alias_mapping_access_authorized': True,
        'condition_identity_restoration_authorized': True,
        'restore_blinded_alias': True,
        'restore_run_id': True,
        'restore_condition_identity': True,
        'freeze_unblinded_response_level_table': True,
        'run_aggregation_authorized': False,
        'condition_level_metric_calculation_authorized': False,
        'primary_arm_comparison_authorized': False,
        'secondary_arm_comparison_authorized': False,
        'bootstrap_inference_authorized': False,
    },
}

prewrite_checks = OrderedDict([
    ('cell7c12_6_artifacts_verified',
     len([x for x in verified_inputs if x['source_cell'] == '7C12']) == 6),
    ('cell7c12_terminal_pass_exact',
     manifest_7c12.get('terminal_decision') == EXPECTED_CELL_7C12_TERMINAL_DECISION),
    ('outcomes_1440',
     int(outcome_pf.metadata.num_rows) == 1440),
    ('outcomes_still_blinded',
     not any(x in outcome_schema for x in ['blinded_alias', 'run_id', 'condition_id', 'condition_name'])),
    ('aggregation_spec_verified',
     agg_record['sha256'] == A004_AGGREGATION_SPEC['sha256']),
    ('routing_map_verified_by_hash_only',
     routing_record['sha256'] == CELL_7C8_ROUTING_MAP['sha256']),
    ('alias_mapping_verified_by_hash_only',
     alias_record['sha256'] == FROZEN_7B4_ALIAS_MAPPING_SHA256),
    ('primary_D_vs_A_frozen',
     aggregation_spec['primary_comparison']['experimental_condition'] == 'D Full-GES'
     and aggregation_spec['primary_comparison']['reference_condition'] == 'A semantic-only'),
    ('bootstrap_2000_frozen',
     aggregation_spec['bootstrap']['replicates'] == 2000),
    ('routing_contents_not_opened_7c13', True),
    ('alias_contents_not_opened_7c13', True),
    ('condition_restoration_not_performed_7c13', True),
    ('run_aggregation_not_performed_7c13', True),
    ('arm_comparison_not_performed_7c13', True),
    ('bootstrap_not_performed_7c13', True),
    ('cell7c14_routing_access_authorized',
     authorization_payload['cell_7c14_scope']['routing_map_access_authorized'] is True),
    ('cell7c14_condition_restoration_authorized',
     authorization_payload['cell_7c14_scope']['condition_identity_restoration_authorized'] is True),
    ('cell7c14_aggregation_still_false',
     authorization_payload['cell_7c14_scope']['run_aggregation_authorized'] is False),
    ('cell7c14_metric_calc_still_false',
     authorization_payload['cell_7c14_scope']['condition_level_metric_calculation_authorized'] is False),
    ('cell7c14_bootstrap_still_false',
     authorization_payload['cell_7c14_scope']['bootstrap_inference_authorized'] is False),
])

failed = [name for name, passed in prewrite_checks.items() if not bool(passed)]
if failed:
    raise RuntimeError(
        'Cell 7C13 authorization QC failed:\\n- ' + '\\n- '.join(failed)
    )

stable_write_json(OUTPUTS['authorization'], authorization_payload)
write_sidecar(OUTPUTS['authorization'])

input_inventory = pd.DataFrame(verified_inputs)
stable_write_csv(OUTPUTS['verified_input_inventory'], input_inventory)
write_sidecar(OUTPUTS['verified_input_inventory'])

terminal_decision = (
    'PASS_STAGE7C13_CELL7C12_BLINDED_RESPONSE_OUTCOMES_REVERIFIED_1440_FROZEN_'
    'A004_ANALYSIS_SPEC_REVERIFIED_CELL7C8_ROUTING_MAP_AND_FROZEN_7B4_ALIAS_'
    'MAPPING_VERIFIED_BY_SHA_WITHOUT_OPENING_CONTENTS_CELL7C14_CONDITION_'
    'RESTORATION_ONLY_AUTHORIZED_NO_RUN_AGGREGATION_CONDITION_LEVEL_METRICS_'
    'ARM_COMPARISON_BOOTSTRAP_OR_INFERENCE'
)

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'checks': {name: bool(value) for name, value in prewrite_checks.items()},
    'passed_checks': len(prewrite_checks),
    'failed_checks': 0,
    'total_checks': len(prewrite_checks),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'upstream_lineage': {
        'cell_7c12_manifest_sha256': CELL_7C12['manifest']['sha256'],
        'cell_7c12_response_level_outcomes_sha256':
            CELL_7C12['response_level_outcomes']['sha256'],
        'cell_7c8_internal_routing_map_sha256':
            CELL_7C8_ROUTING_MAP['sha256'],
        'frozen_7b4_alias_mapping_sha256':
            FROZEN_7B4_ALIAS_MAPPING_SHA256,
        'A004_aggregation_inference_spec_sha256':
            A004_AGGREGATION_SPEC['sha256'],
    },
    'authorization_decision': authorization_decision,
    'terminal_decision': terminal_decision,
    'next_authorized_cell': '7C14',
    'condition_identity_restoration_authorized_in_7c14': True,
    'run_aggregation_authorized_in_7c14': False,
    'condition_level_metric_calculation_authorized_in_7c14': False,
    'arm_comparison_authorized_in_7c14': False,
    'bootstrap_inference_authorized_in_7c14': False,
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
}
stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

for path in OUTPUTS.values():
    if not path.exists() or not sidecar_is_valid(path):
        raise AssertionError(f'Cell 7C13 final readback failed: {path}')

rb_auth = load_json(OUTPUTS['authorization'])
rb_qc = load_json(OUTPUTS['qc'])
rb_manifest = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('next_cell_7c14', rb_manifest.get('next_authorized_cell') == '7C14'),
    ('condition_restoration_true',
     rb_manifest.get('condition_identity_restoration_authorized_in_7c14') is True),
    ('aggregation_false',
     rb_manifest.get('run_aggregation_authorized_in_7c14') is False),
    ('condition_metrics_false',
     rb_manifest.get('condition_level_metric_calculation_authorized_in_7c14') is False),
    ('arm_comparison_false',
     rb_manifest.get('arm_comparison_authorized_in_7c14') is False),
    ('bootstrap_false',
     rb_manifest.get('bootstrap_inference_authorized_in_7c14') is False),
    ('routing_not_opened_7c13',
     rb_auth['cell_7c13_operations']['routing_map_contents_opened'] is False),
    ('alias_not_opened_7c13',
     rb_auth['cell_7c13_operations']['alias_mapping_contents_opened'] is False),
    ('qc_zero_failures', int(rb_qc.get('failed_checks', -1)) == 0),
    ('all_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_rb = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_rb:
    raise RuntimeError(
        'Cell 7C13 final readback QC failed:\\n- ' + '\\n- '.join(failed_rb)
    )

total_checks = len(prewrite_checks) + len(readback_checks)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C13')
print('CONDITION-UNBLINDING AND INTERNAL-ROUTING AUTHORIZATION')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nCELL 7C12 REVERIFICATION')
print(f'Cell 7C12 manifest SHA-256                    : {CELL_7C12["manifest"]["sha256"]}')
print('Cell 7C12 terminal PASS verified              : YES')
print('Frozen blinded response outcomes              : 1,440')
print('Condition identity currently present          : NO')
print('Run ID currently present                      : NO')

print('\\nFROZEN FUTURE-ANALYSIS DESIGN')
print('A004 aggregation/inference spec               : VERIFIED')
print('Primary comparison                            : D Full-GES vs A semantic-only')
print('Mandatory secondary comparisons               : D vs B/C/E/F')
print('Bootstrap                                     : 2,000 paired question-level replicates')

print('\\nROUTING / ALIAS ARTIFACT VERIFICATION')
print('Cell 7C8 internal routing map                  : exact SHA verified')
print('Routing map contents opened in Cell 7C13      : NO')
print('Frozen 7B4 alias mapping                      : exact SHA verified')
print('Alias mapping contents opened in Cell 7C13    : NO')
print(f'Alias mapping artifact                         : {alias_mapping_path}')

print('\\nCELL 7C14 AUTHORIZATION')
print('Open verified internal routing map             : AUTHORIZED')
print('Open verified frozen alias mapping             : AUTHORIZED')
print('Restore blinded alias                         : AUTHORIZED')
print('Restore run_id                                : AUTHORIZED')
print('Restore experimental condition identity       : AUTHORIZED')
print('Freeze unblinded response-level table          : AUTHORIZED')
print('Run aggregation                               : PROHIBITED')
print('Condition-level metric calculation            : PROHIBITED')
print('Arm comparison                                : PROHIBITED')
print('Bootstrap / inference                         : PROHIBITED')

print('\\nCELL 7C13 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_checks}/{total_checks} PASS')

print('\\nNEXT AUTHORIZED CELL')
print('Stage 7C — Cell 7C14                          : condition restoration + unblinded response-table freeze only')
print('Performance calculation                       : NOT YET AUTHORIZED')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C13
CONDITION-UNBLINDING AND INTERNAL-ROUTING AUTHORIZATION
Notebook                                      : 20_GES_Aware_Genomic_RAG_Cell_7C13_Unblinding_and_Routing_Authorization.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nCELL 7C12 REVERIFICATION
Cell 7C12 manifest SHA-256                    : dab5898271ecf7d8304c210afcdf5668e2ada4c6facc86a3d1bb28b23e9de70a
Cell 7C12 terminal PASS verified              : YES
Frozen blinded response outcomes              : 1,440
Condition identity currently present          : NO
Run ID currently present                      : NO
\nFROZEN FUTURE-ANALYSIS DESIGN
A004 aggregation/inference spec               : VERIFIED
Primary comparison                            : D Full-GES vs A semantic-only
Mandatory se